In [14]:
pip install langchain huggingface_hub langchain_community tiktoken rank_bm25 pypdf lancedb langchain-groq

/Users/dhruvdiwakirti/.local/share/uv/python/cpython-3.12-macos-aarch64-none/lib/python3.12/pty.py:95: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid, fd = os.forkpty()
Exception in thread LanceDBBackgroundEventLoop:
Traceback (most recent call last):
  File "/Users/dhruvdiwakirti/.local/share/uv/python/cpython-3.12-macos-aarch64-none/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/Users/dhruvdiwakirti/.local/share/uv/python/cpython-3.12-macos-aarch64-none/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/dhruvdiwakirti/.local/share/uv/python/cpython-3.12-macos-aarch64-none/lib/python3.12/asyncio/base_events.py", line 641, in run_forever
    self


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
from dotenv import load_dotenv

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key

BM25 Retriever - Sparse retriever

Embeddings - Dense retrievers Lancedb

In [16]:
import requests
import os

# Download a public PDF (Attention is All You Need paper from arXiv)
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
pdf_path = "sample.pdf"

response = requests.get(pdf_url)
with open(pdf_path, 'wb') as f:
    f.write(response.content)
print(f"Downloaded PDF to {pdf_path}")

Downloaded PDF to sample.pdf


In [17]:
# Load the PDF
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample.pdf")
pages = loader.load_and_split()
print(f"Loaded {len(pages)} pages from the PDF")

Loaded 16 pages from the PDF


In [18]:
from langchain_community.vectorstores import LanceDB
import lancedb
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import RetrievalQA


In [19]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12411.32it/s]


In [20]:
# Initialize the BM25 retriever
bm25_retriever = BM25Retriever.from_documents(pages)
bm25_retriever.k = 2  # Retrieve top 2 results

print("type of bm25", type(bm25_retriever))

type of bm25 <class 'langchain_community.retrievers.bm25.BM25Retriever'>


In [21]:
db = lancedb.connect("/tmp/lancedb")
table = db.create_table(
    "pandas_docs",
    data=[
        {
            "vector": embedding.embed_query("Hello World"),
            "text": "Hello World",
            "id": "1",
        }
    ],
    mode="overwrite",
)

In [22]:
# Initialize LanceDB retriever
docsearch = LanceDB.from_documents(pages, embedding, connection=db)
retriever_lancedb = docsearch.as_retriever(search_kwargs={"k": 2})

# Initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever_lancedb], weights=[0.2, 0.8]
)

In [23]:
# Example customer query
query = "what is cross attention?"


# Retrieve relevant documents/products
docs = ensemble_retriever.invoke(query)

# Extract and print only the page content from each document
# for doc in docs:
#     print(doc.page_content)

docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'sample.pdf', 'total_pages': 15, 'page': 12, 'page_label': '13'}, page_content='Attention Visualizations\nInput-Input Layer5\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nFigure 3: An example of the attention mechanism follo

## RAGAS Evaluation of the Hybrid RAG

Score the BM25 + LanceDB ensemble retriever (with a Groq LLM for generation) using RAGAS:

- **context_precision** & **context_recall** — quality of what the hybrid retriever pulls back
- **faithfulness** — is the generated answer grounded in the retrieved context?
- **answer_relevancy** — does the answer actually address the question?

Groq (`llama-3.3-70b-versatile`) acts as the judge LLM; the same local HuggingFace embeddings power `answer_relevancy`. No OpenAI key required.

In [24]:
# Already present in this env; uncomment to install elsewhere.
# %pip install -U ragas datasets

In [25]:
# NOTE: ragas 0.4.3 imports `langchain_community.chat_models.vertexai`, which was
# removed in the installed langchain-community. We stub that module so ragas imports
# cleanly (the stub is only touched if you explicitly select a Vertex AI model).
import sys, types

if "langchain_community.chat_models.vertexai" not in sys.modules:
    _stub = types.ModuleType("langchain_community.chat_models.vertexai")
    _stub.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = _stub

from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_56826/3051984823.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_56826/3051984823.py:12: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_56826/3051984823.py:12: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import 

In [26]:
# Generation LLM (Groq) + a simple RAG chain on top of the hybrid retriever
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the answer is not in the context, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

def rag_answer(question: str):
    """Run the hybrid retriever + LLM. Returns (answer, list_of_context_strings)."""
    docs = ensemble_retriever.invoke(question)
    contexts = [d.page_content for d in docs]
    context_text = "\n\n".join(contexts)
    messages = rag_prompt.format_messages(context=context_text, question=question)
    answer = llm.invoke(messages).content
    return answer, contexts

In [27]:
# Evaluation set: questions + reference (ground-truth) answers grounded in the paper
eval_data = [
    {
        "question": "How many layers (N) does the Transformer encoder have?",
        "ground_truth": "The encoder is composed of a stack of N = 6 identical layers.",
    },
    {
        "question": "What is the dimensionality of the model, d_model?",
        "ground_truth": "The model dimensionality d_model is 512.",
    },
    {
        "question": "What activation function is used in the position-wise feed-forward networks?",
        "ground_truth": "A ReLU activation is used between the two linear transformations.",
    },
    {
        "question": "How many attention heads does the model use, and what is the dimension per head?",
        "ground_truth": "The model uses h = 8 parallel attention heads, each with dimension d_k = d_v = d_model / h = 64.",
    },
    {
        "question": "What is multi-head attention?",
        "ground_truth": (
            "Multi-head attention lets the model jointly attend to information from different "
            "representation subspaces at different positions, by running several attention "
            "functions in parallel and concatenating and projecting their outputs."
        ),
    },
]

In [28]:
# Run the hybrid RAG over every question to collect responses + retrieved contexts
samples = []
for row in eval_data:
    answer, contexts = rag_answer(row["question"])
    samples.append({
        "user_input": row["question"],
        "retrieved_contexts": contexts,
        "response": answer,
        "reference": row["ground_truth"],
    })

evaluation_dataset = EvaluationDataset.from_list(samples)
evaluation_dataset.to_pandas()

,user_input,retrieved_contexts,response,reference
0,How many layers (N) does the Transformer encod...,[Figure 1: The Transformer - model architectur...,The Transformer encoder has N = 6 identical la...,The encoder is composed of a stack of N = 6 id...
1,"What is the dimensionality of the model, d_model?",[Table 3: Variations on the Transformer archit...,"The dimensionality of the model, d_model, is 5...",The model dimensionality d_model is 512.
2,What activation function is used in the positi...,[Figure 1: The Transformer - model architectur...,The activation function used in the position-w...,A ReLU activation is used between the two line...
3,"How many attention heads does the model use, a...",[Input-Input Layer5\nThe\nLaw\nwill\nnever\nbe...,The model uses h = 8 parallel attention layers...,"The model uses h = 8 parallel attention heads,..."
4,What is multi-head attention?,[output values. These are concatenated and onc...,Multi-head attention allows the model to joint...,Multi-head attention lets the model jointly at...


In [29]:
# Wrap Groq as the judge LLM and reuse the HF embeddings (for answer_relevancy),
# then score the hybrid RAG with the core RAGAS metrics.
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embedding)

# answer_relevancy normally asks the judge for `strictness` completions in one call
# (n=3). Groq only allows n=1, so we set strictness=1 to avoid a 400 error.
answer_relevancy.strictness = 1

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        context_precision,   # are retrieved chunks relevant to the question?
        context_recall,      # did retrieval cover the reference answer?
        faithfulness,        # is the answer grounded in the retrieved context?
        answer_relevancy,    # does the answer actually address the question?
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(max_workers=2),  # keep low to respect Groq rate limits
)
result

/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_56826/1491913082.py:3: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm)
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_56826/1491913082.py:4: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embedding)
Evaluating: 100%|██████████| 20/20 [05:01<00:00, 15.06s/it]


{'context_precision': 0.7944, 'context_recall': 1.0000, 'faithfulness': 0.9375, 'answer_relevancy': 0.8457}

In [30]:
# Per-sample scores (one row per question)
result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,How many layers (N) does the Transformer encod...,[Figure 1: The Transformer - model architectur...,The Transformer encoder has N = 6 identical la...,The encoder is composed of a stack of N = 6 id...,1.000000,1.0,1.00,0.918862
1,"What is the dimensionality of the model, d_model?",[Table 3: Variations on the Transformer archit...,"The dimensionality of the model, d_model, is 5...",The model dimensionality d_model is 512.,0.805556,1.0,NaN,0.770082
2,What activation function is used in the positi...,[Figure 1: The Transformer - model architectur...,The activation function used in the position-w...,A ReLU activation is used between the two line...,0.833333,1.0,1.00,1.000000
3,"How many attention heads does the model use, a...",[Input-Input Layer5\nThe\nLaw\nwill\nnever\nbe...,The model uses h = 8 parallel attention layers...,"The model uses h = 8 parallel attention heads,...",0.500000,1.0,0.75,0.657402
4,What is multi-head attention?,[output values. These are concatenated and onc...,Multi-head attention allows the model to joint...,Multi-head attention lets the model jointly at...,0.833333,1.0,1.00,0.882112
